# 22 merge picks

22_merge_picks.py

STEP 22 of the Pinatubo FAIR pipeline.

Merge pick indices from:
  - STEP 20: individual-event PHA picks (authoritative / primary)
  - STEP 21: monthly PHA block picks (secondary / provisional)

This step:
• preserves full raw provenance (raw_line, raw_lineno, pha_file)
• deduplicates picks within each event_id
• suppresses monthly picks that duplicate authoritative picks
• emits a unified, FAIR pick table

IMPORTANT SEMANTICS
-------------------
• event_id is the ONLY grouping key
• event_id_source distinguishes authoritative vs provisional groupings
• NO physical events are created here

## Scope and execution

Dataset paths, source priority, tolerances and output naming stay in this Pinatubo workflow.

Review `DATA_TOP` and `SETTINGS`, then run the processing cell. Outputs may overwrite existing products according to the original step. The implementation is in the adjacent local module; no command-line script or subprocess is invoked.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_22_merge_picks import run

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    primary=P['INDIV_PICK_INDEX'],
    secondary=P['MONTHLY_PICK_INDEX'],
    out=P['MERGED_PICK_INDEX'],
    time_tolerance=float(P['PICK_TIME_TOL']),
    report=P['MERGED_PICKS_SUPPRESSED'],
)
display(SETTINGS)


## Run the step

Imports and settings above do not run the analysis. This cell writes the configured products.


In [ ]:
result = run(**SETTINGS)


In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)
